In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")


ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RESUME_PATH = ROOT / "data" / "raw" / "Resume" / "Resume.csv"
JD_PATH = ROOT / "data" / "raw" / "postings.csv"

for path in (RESUME_PATH, JD_PATH):
    assert path.exists(), f"Dataset not found: {path}"

print(f"Resume data: {RESUME_PATH}")
print(f"Job-posting data: {JD_PATH}")

In [ ]:
# Load the two raw datasets. The job-postings file is large (~500 MB), so this may take a little time.
resumes = pd.read_csv(RESUME_PATH)
job_descriptions = pd.read_csv(JD_PATH, low_memory=False)

print(f"Resumes: {resumes.shape[0]:,} rows × {resumes.shape[1]} columns")
print(f"Job descriptions: {job_descriptions.shape[0]:,} rows × {job_descriptions.shape[1]} columns")

In [ ]:
def schema_summary(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    })

print("Resume schema")
display(schema_summary(resumes))

print("Job-description schema")
display(schema_summary(job_descriptions))

In [ ]:

def preview(series, limit=350):
    return series.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip().str[:limit]

resume_sample = resumes.sample(min(5, len(resumes)), random_state=42)[["ID", "Category", "Resume_str"]].copy()
resume_sample["Resume_str"] = preview(resume_sample["Resume_str"])
jd_sample = job_descriptions.sample(min(5, len(job_descriptions)), random_state=42)[["job_id", "title", "company_name", "location", "description"]].copy()
jd_sample["description"] = preview(jd_sample["description"])

display(resume_sample)
display(jd_sample)

In [ ]:

resume_categories = resumes["Category"].fillna("Missing").value_counts()
work_types = job_descriptions["formatted_work_type"].fillna("Missing").value_counts()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.barplot(x=resume_categories.values, y=resume_categories.index, ax=axes[0], palette="viridis")
axes[0].set(title="Resume categories", xlabel="Number of resumes", ylabel="Category")

sns.barplot(x=work_types.values, y=work_types.index, ax=axes[1], palette="magma")
axes[1].set(title="Job-posting employment types", xlabel="Number of postings", ylabel="Work type")
plt.tight_layout()
plt.show()

In [ ]:

quality = pd.DataFrame({
    "dataset": ["resumes", "job_descriptions"],
    "rows": [len(resumes), len(job_descriptions)],
    "duplicate_rows": [resumes.duplicated().sum(), job_descriptions.duplicated().sum()],
    "missing_resume_text_or_category": [
        resumes[["Resume_str", "Category"]].isna().any(axis=1).sum(),
        pd.NA,
    ],
    "missing_title_or_description": [
        pd.NA,
        job_descriptions[["title", "description"]].isna().any(axis=1).sum(),
    ],
})
display(quality)

resume_text_length = resumes["Resume_str"].fillna("").str.len()
jd_text_length = job_descriptions["description"].fillna("").str.len()
garbage_rows = pd.DataFrame({
    "check": [
        "Resumes shorter than 100 characters",
        "JDs with title or description shorter than 20 characters",
        "JDs with non-unique job_id",
    ],
    "flagged_rows": [
        (resume_text_length < 100).sum(),
        ((job_descriptions["title"].fillna("").str.len() < 20) | (jd_text_length < 20)).sum(),
        job_descriptions["job_id"].duplicated(keep=False).sum(),
    ],
})
display(garbage_rows)

In [ ]:
def normalise_text(series):
    return (series.fillna("").astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .str.casefold())

resume_docs = set(normalise_text(resumes["Resume_str"])) - {""}
jd_docs = set(normalise_text(job_descriptions["description"])) - {""}
exact_document_overlap = resume_docs & jd_docs

leakage_report = pd.DataFrame({
    "check": [
        "Exact resume/JD document overlap",
        "Resume IDs duplicated",
        "Job IDs duplicated",
        "Resume category missing",
    ],
    "count": [
        len(exact_document_overlap),
        resumes["ID"].duplicated().sum(),
        job_descriptions["job_id"].duplicated().sum(),
        resumes["Category"].isna().sum(),
    ],
})
display(leakage_report)

if exact_document_overlap:
    print("Warning: inspect overlapping documents before modelling.")
else:
    print("No exact resume/JD document overlap found.")